In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, roc_auc_score
from scipy import stats

# 1. Load data
df = pd.read_csv(r"C:\Users\Lenovo\Desktop\Parkinson's Disease\parkinsons.data")  # adjust path/filename
print(df.shape)
df.head()


(195, 24)


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


In [5]:
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDtypes:\n", df.dtypes.value_counts())
print("\nTarget balance:\n", df['status'].value_counts(normalize=True))

Missing values:
 Series([], dtype: int64)

Duplicate rows: 0

Dtypes:
 float64    22
object      1
int64       1
Name: count, dtype: int64

Target balance:
 status
1    0.753846
0    0.246154
Name: proportion, dtype: float64


In [ ]:
df['subject_id'] = df['name'].apply(lambda x: x.split('_')[2])  
print("Unique subjects:", df['subject_id'].nunique(), "| Total rows:", len(df))
print(df['subject_id'].value_counts())

X = df.drop(columns=['name', 'status', 'subject_id'])
y = df['status']
groups = df['subject_id']

Unique subjects: 32 | Total rows: 195
subject_id
S21    7
S27    7
S35    7
S01    6
S06    6
S07    6
S04    6
S02    6
S10    6
S13    6
S17    6
S16    6
S18    6
S19    6
S08    6
S05    6
S22    6
S20    6
S25    6
S24    6
S31    6
S32    6
S33    6
S26    6
S34    6
S37    6
S39    6
S42    6
S43    6
S44    6
S49    6
S50    6
Name: count, dtype: int64


In [17]:
z_scores = np.abs(stats.zscore(X))
outlier_rows = (z_scores > 4).any(axis=1)
print(f"Rows with |z|>4 in any feature: {outlier_rows.sum()}")
print(df.loc[outlier_rows, 'name'].tolist())

Rows with |z|>4 in any feature: 8
['phon_R01_S19_2', 'phon_R01_S24_4', 'phon_R01_S24_6', 'phon_R01_S35_4', 'phon_R01_S35_6', 'phon_R01_S35_7', 'phon_R01_S49_4', 'phon_R01_S49_5']


In [18]:
corr = X.corr()
high_corr_pairs = [(c1, c2, corr.loc[c1, c2]) 
                    for c1 in corr.columns for c2 in corr.columns 
                    if c1 < c2 and abs(corr.loc[c1, c2]) > 0.9]
high_corr_pairs.sort(key=lambda x: -abs(x[2]))
for c1, c2, v in high_corr_pairs:
    print(f"{c1:25s} <-> {c2:25s}  r={v:.3f}")
print(f"\nTotal pairs > 0.9: {len(high_corr_pairs)}")

Shimmer:APQ3              <-> Shimmer:DDA                r=1.000
Jitter:DDP                <-> MDVP:RAP                   r=1.000
Jitter:DDP                <-> MDVP:Jitter(%)             r=0.990
MDVP:Jitter(%)            <-> MDVP:RAP                   r=0.990
MDVP:Shimmer              <-> Shimmer:DDA                r=0.988
MDVP:Shimmer              <-> Shimmer:APQ3               r=0.988
MDVP:Shimmer              <-> MDVP:Shimmer(dB)           r=0.987
MDVP:Shimmer              <-> Shimmer:APQ5               r=0.983
MDVP:Jitter(%)            <-> MDVP:PPQ                   r=0.974
MDVP:Shimmer(dB)          <-> Shimmer:APQ5               r=0.974
MDVP:Shimmer(dB)          <-> Shimmer:DDA                r=0.963
MDVP:Shimmer(dB)          <-> Shimmer:APQ3               r=0.963
PPE                       <-> spread1                    r=0.962
MDVP:APQ                  <-> MDVP:Shimmer(dB)           r=0.961
Shimmer:APQ5              <-> Shimmer:DDA                r=0.960
Shimmer:APQ3             

In [ ]:
X = df.drop(columns=['name', 'status', 'subject_id'])
y = df['status']
groups = df['subject_id']

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42) # making sure a patient's recordings all stay together; entirely in train, or entirely in test
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

Train class balance:
 status
1    0.763158
0    0.236842
Name: proportion, dtype: float64
Test class balance:
 status
1    0.72093
0    0.27907
Name: proportion, dtype: float64


In [22]:
import numpy as np
import pandas as pd

# 1. Identify features to drop based on correlation > 0.9
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
print(f"Dropping {len(to_drop)} correlated features:")
print(to_drop)

X_reduced = X.drop(columns=to_drop)
print(f"\nFeatures remaining: {X_reduced.shape[1]} (was {X.shape[1]})")

Dropping 11 correlated features:
['MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'PPE']

Features remaining: 11 (was 22)


In [23]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(C=10, gamma=0.01, kernel='rbf', class_weight='balanced', probability=True, random_state=42))
])

gkf = GroupKFold(n_splits=5)
scores_reduced = cross_val_score(pipeline, X_reduced, y, groups=groups, cv=gkf, scoring='roc_auc')

print("Fold AUC scores (reduced features):", scores_reduced)
print("Mean AUC:", scores_reduced.mean(), "| Std:", scores_reduced.std())

print("\n--- Comparison ---")
print("Original (22 features): Mean AUC = 0.6844, Std = 0.1391")
print(f"Reduced ({X_reduced.shape[1]} features): Mean AUC = {scores_reduced.mean():.4f}, Std = {scores_reduced.std():.4f}")

Fold AUC scores (reduced features): [0.7311828  0.65053763 0.8172043  0.63055556 0.86805556]
Mean AUC: 0.7395071684587814 | Std: 0.09210522174901005

--- Comparison ---
Original (22 features): Mean AUC = 0.6844, Std = 0.1391
Reduced (11 features): Mean AUC = 0.7395, Std = 0.0921


In [29]:
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced', probability=True, random_state=42))
])

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 0.001, 0.01, 0.1, 1],
    'svm__kernel': ['rbf']
}

gkf = GroupKFold(n_splits=5)

grid_search = GridSearchCV(
    pipeline, param_grid,
    cv=gkf.split(X_reduced, y, groups=groups),
    scoring='f1_macro',   # <- optimizing for balanced class performance, not just ranking
    n_jobs=-1
)
grid_search.fit(X_reduced, y)

print("Best params:", grid_search.best_params_)
print("Best CV F1-macro:", grid_search.best_score_)

Best params: {'svm__C': 100, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Best CV F1-macro: 0.6363078922642068


In [31]:
from sklearn.model_selection import GroupKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import numpy as np

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(C=100, gamma='scale', kernel='rbf', class_weight='balanced', probability=True, random_state=42))
])

gkf = GroupKFold(n_splits=5)

all_y_true = []
all_y_pred = []

for train_i, test_i in gkf.split(X_reduced, y, groups=groups):
    X_tr, X_te = X_reduced.iloc[train_i], X_reduced.iloc[test_i]
    y_tr, y_te = y.iloc[train_i], y.iloc[test_i]
    
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    
    all_y_true.extend(y_te)
    all_y_pred.extend(y_pred)

print(classification_report(all_y_true, all_y_pred, target_names=['Healthy (0)', 'Diseased (1)']))

              precision    recall  f1-score   support

 Healthy (0)       0.47      0.42      0.44        48
Diseased (1)       0.82      0.84      0.83       147

    accuracy                           0.74       195
   macro avg       0.64      0.63      0.63       195
weighted avg       0.73      0.74      0.73       195

